# Day 2 Code-Along: RAG over real PDFs

**Hands-on AI Workshop · Day 2 · Capstone: Research Paper Summarizer Agent**

Cell numbers (3.2, 5.4, 6.6 …) match the tags on the slides.

| Section | What you do |
|---|---|
| **3. Five ways RAG fails** | Build a naive RAG in 10 lines, then break it five ways on purpose |
| **5. The ingestion pipeline** | Load → clean → split → embed → store, done properly (Chroma + numpy) |
| **6. The RAG query** | Retrieve → threshold → grounded answer with page citations → "I don't know" → eval |
| **Cheatsheet** | Today on one page |

**If you fall behind or your runtime restarts:** run the **⏩ catch-up cell** (3.0, 5.0, 6.0) at the start
of the section you want to rejoin. **If a cell fails:** post the error in the team chat; a TA will help.

> 📄 Today's papers are **real** arXiv papers, redistributed under CC BY 4.0 (see `MANIFEST.md`):
> *Mistral 7B*, *Ragas*, *vLLM / PagedAttention* and *Dense X Retrieval*.
> The small "edge-case" PDFs (two-column, image-only) are **fictional**, generated for the workshop.

---
# Section 3: Five ways RAG fails (on purpose)

First a naive RAG in ten lines. It works on an easy question. Then we break it five ways, each one fixed later today.

In [ ]:
# ⏩ 3.0 CATCH-UP: run ONLY if you joined late or your runtime restarted.
# It recreates everything earlier sections defined, so you can follow along from here.
import importlib.util
if importlib.util.find_spec("paper_agent") is None:      # Colab: install the course package (about 1 minute)
    !pip install -q "paper-agent[rag] @ git+https://github.com/mari-muthu-k/agent-to-production@main#subdirectory=paper-summarizer"

import logging, os, re, time, json
import numpy as np
import openai
from paper_agent.config import load_env
from paper_agent.llm_client import LLMClient, LLMConfig, summarize_calls
from paper_agent.ingest.embeddings import OpenAIEmbedder
from paper_agent.openrouter import make_client, is_openrouter, REASONING_OFF

# Colab Secrets (🔑) or .env: LLM_API_KEY, LLM_BASE_URL, LLM_MODEL, EMBED_MODEL (+ optional LLM_FALLBACK_MODEL)
# OpenRouter: LLM_BASE_URL=https://openrouter.ai/api/v1, e.g. EMBED_MODEL=nvidia/nemotron-3-embed-1b:free
env = load_env(required=("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "EMBED_MODEL"))
MODEL, EMBED_MODEL = env["LLM_MODEL"], env["EMBED_MODEL"]
PRICE_IN_PER_1M, PRICE_OUT_PER_1M = 0.50, 2.00   # USD per 1M tokens (illustrative: the instructor gives real ones)
PRICE_EMBED_PER_1M = 0.02
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s", force=True)

client = make_client(env["LLM_API_KEY"], env["LLM_BASE_URL"])     # OpenAI SDK; works with OpenRouter
REASONING = REASONING_OFF if is_openrouter() else None   # short grounded answers don't need long thinking
llm = LLMClient(LLMConfig(model=MODEL, fallback_model=env.get("LLM_FALLBACK_MODEL") or None,
                          price_in_per_1m=PRICE_IN_PER_1M, price_out_per_1m=PRICE_OUT_PER_1M,
                          max_tokens_param=env.get("LLM_MAX_TOKENS_PARAM") or "max_tokens",
                          reasoning=REASONING), client=client)
embedder = OpenAIEmbedder(EMBED_MODEL, client=client, price_per_1m=PRICE_EMBED_PER_1M)

from paper_agent.papers import fetch_paper, paper_ids, paper_info, edge_case_pdf
PAPERS = {pid: fetch_paper(pid) for pid in paper_ids()}      # 4 real papers, CC BY 4.0
print('✅ caught up: ready for Section 3')

### 3.1 Setup

Colab **Secrets** (🔑, Notebook access ON): `LLM_API_KEY`, `LLM_BASE_URL`, `LLM_MODEL`, **`EMBED_MODEL`** (new today).
The first run installs the course package (about a minute).

> **OpenRouter:** `LLM_BASE_URL` = `https://openrouter.ai/api/v1`; for embeddings use e.g.
> `nvidia/nemotron-3-embed-1b:free` or `openai/text-embedding-3-small`. Free (`:free`) models allow
> 20 requests/minute and 50/day without credits, so today's notebook batches its embedding calls.

In [ ]:
import importlib.util
if importlib.util.find_spec("paper_agent") is None:      # Colab: install the course package (about 1 minute)
    !pip install -q "paper-agent[rag] @ git+https://github.com/mari-muthu-k/agent-to-production@main#subdirectory=paper-summarizer"

import logging, os, re, time, json
import numpy as np
import openai
from paper_agent.config import load_env
from paper_agent.llm_client import LLMClient, LLMConfig, summarize_calls
from paper_agent.ingest.embeddings import OpenAIEmbedder
from paper_agent.openrouter import make_client, is_openrouter, REASONING_OFF

# Colab Secrets (🔑) or .env: LLM_API_KEY, LLM_BASE_URL, LLM_MODEL, EMBED_MODEL (+ optional LLM_FALLBACK_MODEL)
# OpenRouter: LLM_BASE_URL=https://openrouter.ai/api/v1, e.g. EMBED_MODEL=nvidia/nemotron-3-embed-1b:free
env = load_env(required=("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "EMBED_MODEL"))
MODEL, EMBED_MODEL = env["LLM_MODEL"], env["EMBED_MODEL"]
PRICE_IN_PER_1M, PRICE_OUT_PER_1M = 0.50, 2.00   # USD per 1M tokens (illustrative: the instructor gives real ones)
PRICE_EMBED_PER_1M = 0.02
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s", force=True)

client = make_client(env["LLM_API_KEY"], env["LLM_BASE_URL"])     # OpenAI SDK; works with OpenRouter
REASONING = REASONING_OFF if is_openrouter() else None   # short grounded answers don't need long thinking
llm = LLMClient(LLMConfig(model=MODEL, fallback_model=env.get("LLM_FALLBACK_MODEL") or None,
                          price_in_per_1m=PRICE_IN_PER_1M, price_out_per_1m=PRICE_OUT_PER_1M,
                          max_tokens_param=env.get("LLM_MAX_TOKENS_PARAM") or "max_tokens",
                          reasoning=REASONING), client=client)
embedder = OpenAIEmbedder(EMBED_MODEL, client=client, price_per_1m=PRICE_EMBED_PER_1M)

from paper_agent.papers import fetch_paper, paper_ids, paper_info, edge_case_pdf
PAPERS = {pid: fetch_paper(pid) for pid in paper_ids()}      # 4 real papers, CC BY 4.0

for pid in PAPERS:
    info = paper_info(pid)
    print(f"{pid:11} {info['pages']:>2} pages  {info['title'][:58]}  ({info['license']})")
print("✅ ready | chat:", MODEL, "| embeddings:", EMBED_MODEL)

### 3.2 A naive RAG in ten lines

Fixed 300-character chunks, top-3 by cosine, everything into the prompt.

In [ ]:
from paper_agent.ingest.loaders import load_pdf
NAIVE_PROMPT = 'Answer the question using the context.'

def naive_chunks(text: str, size: int = 300) -> list:
    return [text[i:i + size] for i in range(0, len(text), size)]

def naive_search(question: str, chunks: list, matrix: np.ndarray, embed, k: int = 3) -> list:
    q = embed([question])[0]
    scores = matrix @ q
    order = np.argsort(-scores)[:k]
    return [(float(scores[i]), chunks[i]) for i in order]

def naive_answer(question: str, context_chunks: list, client, model: str, max_tokens: int = 200) -> str:
    context = "\n\n".join(f"<chunk>{c}</chunk>" for c in context_chunks)
    r = client.chat.completions.create(model=model, max_tokens=max_tokens, temperature=0, messages=[
        {"role": "system", "content": NAIVE_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}])
    return r.choices[0].message.content

In [ ]:
text = load_pdf(PAPERS["mistral-7b"], layout="naive").text      # read every page straight across
naive = naive_chunks(text, size=300)
naive_matrix = embedder.embed(naive)
question = "How does the rolling buffer cache limit memory use?"
hits = naive_search(question, naive, naive_matrix, embedder.embed, k=3)
for score, chunk in hits:
    print(f"{score:.2f}  {chunk[:90]!r}")
print("\n🤖", naive_answer(question, [c for _, c in hits], client, MODEL))

It probably worked. **That's the trap.** Five ways to break it:

### 3.3 💥 Failure 1: bad chunking (and bad parsing before it)

In [ ]:
for left, right in list(zip(naive, naive[1:]))[:4]:          # where the 300-char windows cut
    print(f"...{left[-40:]!r}  ✂️  {right[:40]!r}...")
print("\nRagas page 2, read straight across the page (it has two columns):")
print("\n".join(load_pdf(PAPERS["ragas"], layout="naive").pages[1].text.splitlines()[:5]))

Words, numbers and sentences cut in half; two columns braided together. **Fixed in 5.1–5.3.**

### 3.4 💥 Failure 2: embedding mismatch

In [ ]:
from paper_agent.ingest.embeddings import HashingEmbedder
other_model = HashingEmbedder(dim=naive_matrix.shape[1])     # a different "embedding model", same size
same = naive_search(question, naive, naive_matrix, embedder.embed, k=1)[0]
mixed = naive_search(question, naive, naive_matrix, other_model.embed, k=1)[0]
print(f"query embedded with the SAME model     : {same[0]:.2f}  {same[1][:70]!r}")
print(f"query embedded with a DIFFERENT model  : {mixed[0]:.2f}  {mixed[1][:70]!r}")
print("\nNo error, no warning: just quietly wrong results.")

Vectors from different models live in different spaces. **Fixed in 5.5:** stores remember their model and refuse others.

### 3.5 💥 Failure 3: retrieval noise

In [ ]:
for score, chunk in naive_search("What is the best pizza topping?", naive, naive_matrix, embedder.embed, k=3):
    print(f"{score:.2f}  {chunk[:80]!r}")
print("\nTop-k always returns k chunks, even when none of them is relevant. All go into the prompt.")

**Fixed in 6.2:** a score threshold.

### 3.6 💥 Failure 4: context overflow ("just send everything")

In [ ]:
from paper_agent.tokens import count_tokens
everything = "\n\n".join(load_pdf(p, layout="naive").text for p in PAPERS.values())
n = count_tokens(everything)
print(f"all 4 papers: {n:,} tokens  ->  ${n * PRICE_IN_PER_1M / 1e6:.4f} per question, "
      f"${n * PRICE_IN_PER_1M / 1e6 * 20 * 200:,.2f} for 20 questions x 200 people")
try:
    r = client.chat.completions.create(model=MODEL, max_tokens=60, messages=[
        {"role": "user", "content": everything + "\n\nQuestion: " + question}])
    print(f"Accepted ({r.usage.prompt_tokens:,} prompt tokens). It fits this model, but you pay for all of it "
          "on every question, and answers buried mid-context are often missed.")
except openai.BadRequestError as e:
    print("💥", type(e).__name__, "-", e.message[:170])

**Fixed in 6.3:** send only the best chunks that fit a token budget.

### 3.7 💥 Failure 5: hallucination

In [ ]:
unanswerable = "Which GPU cloud provider sponsored the training run?"
hits = naive_search(unanswerable, naive, naive_matrix, embedder.embed, k=3)
answer = naive_answer(unanswerable, [c for _, c in hits], client, MODEL)
print("🤖", answer)
numbers = re.findall(r"\d[\d.,]*\d|\d", answer)
print("\nnumbers in the answer:", numbers, "| actually in the paper:", [x for x in numbers if x in text])

The prompt never said *"you may say you don't know"*, so a confident answer is the path of least resistance.
**Fixed in 6.4–6.6:** an "insufficient evidence" path, page citations checked against what was retrieved,
and a number check.

| # | Failure | Symptom | Fix |
|---|---|---|---|
| 1 | Bad chunking / parsing | Answers cut in half, columns mixed | 5.1 column-aware loader, 5.3 sentence-aware splitter |
| 2 | Embedding mismatch | Plausible but wrong chunks, no error | 5.5 one model per index, enforced |
| 3 | Retrieval noise | Irrelevant chunks in the prompt | 6.2 score threshold |
| 4 | Context overflow | 400 error, or cost and buried answers | 6.3 top-k within a token budget |
| 5 | Hallucination | Confident made-up facts | 6.4–6.6 "I don't know", citation guard, number check |

---
# Section 5: The ingestion pipeline

Load → clean → split → embed → store. Done once per paper.

In [ ]:
# ⏩ 5.0 CATCH-UP: run ONLY if you joined late or your runtime restarted.
# It recreates everything earlier sections defined, so you can follow along from here.
import importlib.util
if importlib.util.find_spec("paper_agent") is None:      # Colab: install the course package (about 1 minute)
    !pip install -q "paper-agent[rag] @ git+https://github.com/mari-muthu-k/agent-to-production@main#subdirectory=paper-summarizer"

import logging, os, re, time, json
import numpy as np
import openai
from paper_agent.config import load_env
from paper_agent.llm_client import LLMClient, LLMConfig, summarize_calls
from paper_agent.ingest.embeddings import OpenAIEmbedder
from paper_agent.openrouter import make_client, is_openrouter, REASONING_OFF

# Colab Secrets (🔑) or .env: LLM_API_KEY, LLM_BASE_URL, LLM_MODEL, EMBED_MODEL (+ optional LLM_FALLBACK_MODEL)
# OpenRouter: LLM_BASE_URL=https://openrouter.ai/api/v1, e.g. EMBED_MODEL=nvidia/nemotron-3-embed-1b:free
env = load_env(required=("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "EMBED_MODEL"))
MODEL, EMBED_MODEL = env["LLM_MODEL"], env["EMBED_MODEL"]
PRICE_IN_PER_1M, PRICE_OUT_PER_1M = 0.50, 2.00   # USD per 1M tokens (illustrative: the instructor gives real ones)
PRICE_EMBED_PER_1M = 0.02
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s", force=True)

client = make_client(env["LLM_API_KEY"], env["LLM_BASE_URL"])     # OpenAI SDK; works with OpenRouter
REASONING = REASONING_OFF if is_openrouter() else None   # short grounded answers don't need long thinking
llm = LLMClient(LLMConfig(model=MODEL, fallback_model=env.get("LLM_FALLBACK_MODEL") or None,
                          price_in_per_1m=PRICE_IN_PER_1M, price_out_per_1m=PRICE_OUT_PER_1M,
                          max_tokens_param=env.get("LLM_MAX_TOKENS_PARAM") or "max_tokens",
                          reasoning=REASONING), client=client)
embedder = OpenAIEmbedder(EMBED_MODEL, client=client, price_per_1m=PRICE_EMBED_PER_1M)

from paper_agent.papers import fetch_paper, paper_ids, paper_info, edge_case_pdf
PAPERS = {pid: fetch_paper(pid) for pid in paper_ids()}      # 4 real papers, CC BY 4.0
print('✅ caught up: ready for Section 5')

### 5.1 Loaders: read the columns in the right order

In [ ]:
from paper_agent.ingest.loaders import load_pdf
doc = load_pdf(PAPERS["ragas"])                                  # column-aware (the default)
print(len(doc.pages), "pages | columns per page:", [p.columns for p in doc.pages])
print("\nRagas page 2, column-aware:\n" + "\n".join(doc.pages[1].text.splitlines()[:5]))
for name in ("two_column", "image_only"):                       # FICTIONAL edge-case PDFs
    d = load_pdf(edge_case_pdf(name))
    print(f"\n{name}: columns={[p.columns for p in d.pages]} warnings={d.warnings}")

An image-only page has no text to extract: we **say so** instead of silently indexing nothing.

### 5.2 Clean up and find the sections (for citations)

In [ ]:
from paper_agent.ingest.cleaning import to_blocks
blocks = to_blocks(doc.pages)              # drops headers/footers, page numbers, references
sections = []
for b in blocks:
    if not sections or sections[-1][0] != b.section:
        sections.append((b.section, b.page))
print(" · ".join(f"{s} (p.{p})" for s, p in sections))

### 5.3 Split: 500–1,000 characters, at sentence boundaries, with overlap (LangChain)

In [ ]:
CHUNK_SIZE, CHUNK_OVERLAP = 800, 120

def sentence_splitter(chunk_size: int = CHUNK_SIZE, chunk_overlap: int = CHUNK_OVERLAP):
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", "? ", "! ", "; ", ", ", " ", ""],
        keep_separator="end",          # the full stop stays with its sentence
        add_start_index=True,
    )

sample = blocks[3].text
for piece in sentence_splitter(chunk_size=300, chunk_overlap=60).split_text(sample)[:3]:
    print(repr(piece), "\n")

In [ ]:
from paper_agent.ingest.splitters import fixed_chunks, sentence_chunks, chunk_stats
fixed = fixed_chunks(blocks, "ragas", size=300)
chunks = sentence_chunks(blocks, "ragas")          # the same splitter, per section, tracking pages
print("fixed 300  :", chunk_stats(fixed))
print("sentence   :", chunk_stats(chunks))
c = chunks[4]
print(f"\n[{c.id}] section={c.section} page={c.page}\n{c.text}")

Every chunk now knows its **section and page**: that is what citations will point to.

### 5.4 Embed

In [ ]:
vectors = embedder.embed([c.text for c in chunks])
print("matrix:", vectors.shape, "| row lengths:", np.round(np.linalg.norm(vectors[:3], axis=1), 3),
      "| tokens embedded so far:", embedder.tokens_used)

### 5.5 Store and search: ✏️ TODO 1 (2 lines)

With unit-length rows, cosine similarity is a dot product, so one matrix-vector product scores every chunk.
Return the `k` best `(row_index, score)` pairs, best first.

In [ ]:
def top_k(matrix: np.ndarray, query: np.ndarray, k: int) -> list:
    """Indices and scores of the k rows most similar to `query` (all rows L2-normalised)."""
    # ✏️ TODO 1: replace the line below.
    #   1. scores for every row at once: a matrix-vector product
    #   2. the k best indices, highest score first (np.argsort sorts ascending)
    return [(i, float(matrix[i] @ query)) for i in range(k)]

In [ ]:
m = np.eye(4, dtype=np.float32)
q = np.array([0.1, 0.9, 0.3, 0.0], dtype=np.float32)
got = [i for i, _ in top_k(m, q, 2)]
assert got == [1, 2], f"TODO 1: expected rows [1, 2] (best first), got {got}"
assert len(top_k(m, q, 3)) == 3
print("✅ TODO 1: top_k returns the most similar rows, best first")

<details><summary>🆘 Stuck? Hint</summary>

`scores = matrix @ query`, then `idx = np.argsort(-scores)[:k]`, then return `[(int(i), float(scores[i])) for i in idx]`.
</details>

🆘 **Rescue cell:** tests failing, or you fell behind? Run the next cell, then carry on.

In [ ]:
# 🆘 RESCUE: the finished version. Running it is always safe.
def top_k(matrix: np.ndarray, query: np.ndarray, k: int) -> list:
    """Indices and scores of the k rows most similar to `query` (all rows L2-normalised)."""
    scores = matrix @ query
    idx = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in idx]

In [ ]:
def search(question, k=3):
    q = embedder.embed([question])[0]
    return [(score, chunks[i]) for i, score in top_k(vectors, q, k)]

for score, c in search("Which three quality aspects of a RAG system does Ragas evaluate?"):
    print(f"{score:.2f}  {c.section} p.{c.page}  {c.text[:80]!r}")

That is the whole mechanism of a vector database. Now the same thing in **Chroma**, persisted to disk, for all four papers. The store remembers its embedding model, which fixes failure 2.

In [ ]:
from paper_agent.ingest.pipeline import ingest_pdf
from paper_agent.ingest.store import ChromaStore
store = ChromaStore(EMBED_MODEL, path="data/vectorstore")         # one collection per embedding model
for pid, path in PAPERS.items():
    report, _ = ingest_pdf(path, embedder, store, paper_id=pid)
    print(report)
print(f"\n{len(store)} chunks in Chroma collection '{store.collection.name}' (saved in data/vectorstore)")

In [ ]:
from paper_agent.ingest.embeddings import EmbeddingMismatchError, HashingEmbedder
try:
    store.search(HashingEmbedder().embed(["rolling buffer cache"])[0], embed_model="hashing-local-256")
except EmbeddingMismatchError as e:
    print("🛡", e)

### 5.6 Fixed vs semantic chunking, side by side

Semantic chunking starts a new chunk where the topic shifts: where the next sentence's embedding is least similar to the current one.

In [ ]:
from paper_agent.ingest.splitters import semantic_chunks
strategies = {"fixed 300": fixed, "sentence 800": chunks,
              "semantic": semantic_chunks(blocks, "ragas", embedder.embed)}
for name, cs in strategies.items():
    s = chunk_stats(cs)
    print(f"{name:13} {s['chunks']:>3} chunks  median {s['median_chars']:>4} chars  "
          f"{s['cut_mid_sentence']:>2} cut mid-sentence")
definition = ("Faithfulness refers to the idea that the answer should be grounded in the given context. "
              "This is important to avoid hallucinations, and to ensure that the retrieved context can act "
              "as a justification for the generated answer.")
print("\nWhere does the definition of faithfulness land?")
for name, cs in strategies.items():
    whole = [c for c in cs if definition in c.text]
    if whole:
        c = whole[0]
        i = c.text.index(definition)
        print(f"[{name}] whole, in one chunk (p.{c.page}): {c.text[i:i + 90]!r}...")
    else:
        print(f"[{name}] ✂️ split across two chunks: neither half answers 'what is faithfulness?'")

Semantic chunks follow the argument but cost an embedding per sentence at ingest time. Sentence-aware fixed-size chunks are the usual default.

---
# Section 6: The RAG query

Retrieve → keep only confident matches → grounded answer with page citations → verify → or say "I don't know".

In [ ]:
# ⏩ 6.0 CATCH-UP: run ONLY if you joined late or your runtime restarted.
# It recreates everything earlier sections defined, so you can follow along from here.
import importlib.util
if importlib.util.find_spec("paper_agent") is None:      # Colab: install the course package (about 1 minute)
    !pip install -q "paper-agent[rag] @ git+https://github.com/mari-muthu-k/agent-to-production@main#subdirectory=paper-summarizer"

import logging, os, re, time, json
import numpy as np
import openai
from paper_agent.config import load_env
from paper_agent.llm_client import LLMClient, LLMConfig, summarize_calls
from paper_agent.ingest.embeddings import OpenAIEmbedder
from paper_agent.openrouter import make_client, is_openrouter, REASONING_OFF

# Colab Secrets (🔑) or .env: LLM_API_KEY, LLM_BASE_URL, LLM_MODEL, EMBED_MODEL (+ optional LLM_FALLBACK_MODEL)
# OpenRouter: LLM_BASE_URL=https://openrouter.ai/api/v1, e.g. EMBED_MODEL=nvidia/nemotron-3-embed-1b:free
env = load_env(required=("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "EMBED_MODEL"))
MODEL, EMBED_MODEL = env["LLM_MODEL"], env["EMBED_MODEL"]
PRICE_IN_PER_1M, PRICE_OUT_PER_1M = 0.50, 2.00   # USD per 1M tokens (illustrative: the instructor gives real ones)
PRICE_EMBED_PER_1M = 0.02
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s", force=True)

client = make_client(env["LLM_API_KEY"], env["LLM_BASE_URL"])     # OpenAI SDK; works with OpenRouter
REASONING = REASONING_OFF if is_openrouter() else None   # short grounded answers don't need long thinking
llm = LLMClient(LLMConfig(model=MODEL, fallback_model=env.get("LLM_FALLBACK_MODEL") or None,
                          price_in_per_1m=PRICE_IN_PER_1M, price_out_per_1m=PRICE_OUT_PER_1M,
                          max_tokens_param=env.get("LLM_MAX_TOKENS_PARAM") or "max_tokens",
                          reasoning=REASONING), client=client)
embedder = OpenAIEmbedder(EMBED_MODEL, client=client, price_per_1m=PRICE_EMBED_PER_1M)

from paper_agent.papers import fetch_paper, paper_ids, paper_info, edge_case_pdf
PAPERS = {pid: fetch_paper(pid) for pid in paper_ids()}      # 4 real papers, CC BY 4.0

def top_k(matrix: np.ndarray, query: np.ndarray, k: int) -> list:
    """Indices and scores of the k rows most similar to `query` (all rows L2-normalised)."""
    scores = matrix @ query
    idx = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in idx]

from paper_agent.ingest.pipeline import ingest_pdf
from paper_agent.ingest.store import ChromaStore
store = ChromaStore(EMBED_MODEL, path="data/vectorstore")         # one collection per embedding model
for pid, path in PAPERS.items():
    report, _ = ingest_pdf(path, embedder, store, paper_id=pid)
    print(report)
print(f"\n{len(store)} chunks in Chroma collection '{store.collection.name}' (saved in data/vectorstore)")
print('✅ caught up: ready for Section 6')

### 6.1 Retrieve across all four papers

In [ ]:
from paper_agent.rag.retrieve import Retriever, show_hits
retriever = Retriever(store, embedder, k=5)
show_hits(retriever.search("How does vLLM share KV cache blocks between parallel samples?"))

### 6.2 How sure is "sure"? Calibrate a score threshold

In [ ]:
from paper_agent.rag.evals import golden_questions
on_topic = [g["question"] for g in golden_questions()]
off_topic = ["What is the best pizza topping?", "Who won the 2018 football World Cup?", "How do I renew a passport?"]
embedder.embed(on_topic + off_topic)          # one request for all 13 questions (free tiers count requests)
best = lambda qs: [retriever.search(q, k=1)[0].score for q in qs]
on, off = best(on_topic), best(off_topic)
print("best score, questions the papers answer :", np.round(sorted(on), 2))
print("best score, questions they don't        :", np.round(sorted(off), 2))
MIN_SCORE = round((float(np.percentile(on, 25)) + max(off)) / 2, 2)
print("\nthreshold between the groups -> MIN_SCORE =", MIN_SCORE)
if min(on) <= max(off):
    print("⚠ the groups overlap: any threshold trades missed answers for noise. Better embeddings separate them.")

### ✏️ TODO 2 (1 line): drop weak matches

In [ ]:
def keep_confident(hits: list, min_score: float) -> list:
    """Drop hits whose similarity is below the threshold."""
    # ✏️ TODO 2: keep only the hits with h.score >= min_score
    return hits

In [ ]:
from paper_agent.ingest.store import Hit
from paper_agent.ingest.splitters import Chunk
fake = [Hit(Chunk(f"x:{i}", "x", "text", "results", 1, [1]), s) for i, s in enumerate([0.9, 0.5, 0.2])]
kept = [h.score for h in keep_confident(fake, 0.4)]
assert kept == [0.9, 0.5], f"TODO 2: expected [0.9, 0.5], got {kept}"
assert keep_confident(fake, 0.95) == []
print("✅ TODO 2: weak matches are dropped")

<details><summary>🆘 Stuck? Hint</summary>

`return [h for h in hits if h.score >= min_score]`
</details>

🆘 **Rescue cell:** tests failing, or you fell behind? Run the next cell, then carry on.

In [ ]:
# 🆘 RESCUE: the finished version. Running it is always safe.
def keep_confident(hits: list, min_score: float) -> list:
    """Drop hits whose similarity is below the threshold."""
    return [h for h in hits if h.score >= min_score]

### 6.3 The grounded prompt: chunks as data, (section, page) on every chunk, a way out

In [ ]:
from paper_agent.rag.answer import QA_PROMPT, rag_messages, fit_to_budget, allowed_citations
print(QA_PROMPT)
print(rag_messages("How was the WikiEval dataset built?",
                   retriever.search("How was the WikiEval dataset built?", k=1))[1]["content"][:400])

### 6.4 Citation guard: ✏️ TODO 3 (1 line)

Day 1 checked section ids. Now a citation is a **(section, page)** pair, and it must belong to a chunk
we actually retrieved and sent. `allowed_citations(hits)` gives the set of allowed pairs.

In [ ]:
from paper_agent.schemas import GroundedAnswer

class PageCitationError(Exception):
    pass

def check_page_citations(result: GroundedAnswer, hits: list) -> None:
    """Every cited (section, page) must belong to a chunk we actually sent."""
    unknown = set()   # ✏️ TODO 3: the cited (section, page) pairs that are NOT in allowed_citations(hits)
    if unknown:
        raise PageCitationError(f"cites (section, page) pairs that were not retrieved: {sorted(unknown)}")

In [ ]:
from paper_agent.ingest.store import Hit
from paper_agent.ingest.splitters import Chunk
sent = [Hit(Chunk("p:1", "p", "Recall rose to 74%.", "results", 6, [6]), 0.8)]
ok = GroundedAnswer(status="answered", answer="Recall rose to 74%.", confidence="high",
                    citations=[{"section": "results", "page": 6}])
bad = GroundedAnswer(status="answered", answer="Recall rose to 74%.", confidence="high",
                     citations=[{"section": "results", "page": 9}])
check_page_citations(ok, sent)
try:
    check_page_citations(bad, sent)
    raise AssertionError("TODO 3: page 9 was never retrieved, but no error was raised")
except PageCitationError as e:
    print("✅ TODO 3:", e)

<details><summary>🆘 Stuck? Hint</summary>

`unknown = {(c.section, c.page) for c in result.citations} - allowed_citations(hits)`
</details>

🆘 **Rescue cell:** tests failing, or you fell behind? Run the next cell, then carry on.

In [ ]:
# 🆘 RESCUE: the finished version. Running it is always safe.
from paper_agent.schemas import GroundedAnswer

class PageCitationError(Exception):
    pass

def check_page_citations(result: GroundedAnswer, hits: list) -> None:
    """Every cited (section, page) must belong to a chunk we actually sent."""
    unknown = {(c.section, c.page) for c in result.citations} - allowed_citations(hits)
    if unknown:
        raise PageCitationError(f"cites (section, page) pairs that were not retrieved: {sorted(unknown)}")

### 6.5 Put it together: `ask()`

In [ ]:
from paper_agent.rag.answer import RAGAnswer, NOT_FOUND, unsupported_numbers

def ask(question, k=5, max_context_tokens=3000):
    hits = keep_confident(retriever.search(question, k=k), MIN_SCORE)
    if not hits:                                    # nothing relevant: say so, and save the call
        return RAGAnswer(question, "insufficient_evidence", NOT_FOUND, llm_called=False)
    hits = fit_to_budget(hits, max_context_tokens)  # best chunks first, within a token budget
    result = llm.chat_structured(rag_messages(question, hits), GroundedAnswer, max_tokens=800)
    check_page_citations(result, hits)              # TODO 3: cite only what was retrieved
    missing = unsupported_numbers(result, hits) if result.status == "answered" else []
    warnings = [f"numbers not found in the cited pages: {missing}"] if missing else []
    return RAGAnswer(question, result.status, result.answer, result.citations, result.confidence, hits, warnings)

for q in ["How does the rolling buffer cache limit memory use?",
          "By how much does vLLM improve LLM serving throughput?"]:
    ask(q).show()

`paper_agent.rag.answer.answer_question` is the same function, plus one repair attempt on a bad citation (the Day 1 pattern).

### 6.6 "I don't know" is a feature

In [ ]:
ask("What is the best pizza topping?").show()                                    # stopped by the threshold: no LLM call
ask("Which GPU cloud provider sponsored the Mistral 7B training run?").show()   # relevant-looking chunks, no answer
bad = GroundedAnswer(status="answered", answer="It used 512 A100 GPUs.", confidence="high",
                     citations=[{"section": "results", "page": 3}])
hits = retriever.search("How does the rolling buffer cache limit memory use?", k=3)
try:
    check_page_citations(bad, hits)
except PageCitationError as e:
    print("\n🛡 citation guard:", e)

### 6.7 Measure retrieval: hit rate @k on 10 golden questions

In [ ]:
from paper_agent.rag.evals import hit_rate_at_k
hit_rate_at_k(retriever, golden_questions(), k=5).show()
for k in (1, 3):
    print(f"hit rate @{k}: {hit_rate_at_k(retriever, golden_questions(), k=k).hit_rate:.0%}")

If the right page never reaches the prompt, no prompt can save the answer. Measure retrieval on its own, every time you change chunking, embeddings or k.

### 6.8 What did today cost?

In [ ]:
import pprint
pprint.pprint(summarize_calls())
print(f"embedding tokens: {embedder.tokens_used:,} (≈ ${embedder.tokens_used * PRICE_EMBED_PER_1M / 1e6:.4f})")

---
# RAG Cheatsheet

| Area | Rules | Practised in |
|---|---|---|
| **Parsing** | Check columns, headers and image-only pages · Say what you could not read | 3.3, 5.1 |
| **Chunking** | 500–1,000 chars at sentence boundaries, with overlap · Keep section + page on every chunk | 5.2, 5.3, 5.6 |
| **Embeddings** | One model per index, enforced · Re-embed everything when you change model | 3.4, 5.5 |
| **Retrieval** | Top-k **and** a calibrated threshold · Measure hit rate @k on golden questions | 3.5, 6.2, 6.7 |
| **Context** | Best chunks first, within a token budget · Never "send the whole paper" | 3.6, 6.3 |
| **Answers** | Chunks are data, never instructions · Cite (section, page) · Verify citations against what was sent · Let the model say "insufficient evidence" | 3.7, 6.4–6.6 |

---
### Extensions (homework)

**E1. Compare strategies with numbers.** Index all four papers with fixed, sentence and semantic chunks and
compare `hit_rate_at_k` for each.

**E2. Hybrid search.** Add a keyword score (count of shared words) to the cosine score. Which golden
questions improve?

**E3. Re-ranking.** Retrieve 20, then ask the LLM to pick the best 5. What does it cost per question?

**E4. Your own paper.** Upload a PDF to Colab, `ingest_pdf` it, and write three golden questions for it.

**Tomorrow (Day 3):** the agent: LangChain tools (`search_paper`, `get_section`, `define_term`, `save_note`),
an agent loop with limits, and safety guards against instructions hidden inside PDFs.